# Tourism Demand Forecasting

## Project Overview

Forecasts **daily tourism visitor count** (thousands) over a 14-day horizon. Synthetic data spans 2 years with strong seasonal patterns, weekend surges, and holiday peaks.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily visitor counts, predict the next 14 days. Tourism boards and destination managers need demand forecasts for infrastructure planning, marketing budgets, and resource allocation.

## Why This Project Matters

Tourism is a major economic sector (10%+ of global GDP). Destinations that accurately forecast visitor flows can optimize infrastructure, manage environmental impact, reduce overcrowding, and target marketing spend effectively.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily visitor counts (thousands)
- Strong seasonal pattern (summer peak, winter trough)
- Weekend surges (day-trippers and short breaks)
- Holiday season peaks
- Growth trend (destination popularity)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `visitors_k` | Daily visitor count (thousands) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'visitors_k'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 20 + 0.01 * t  # growing popularity
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = -2  # weekday lower
    elif dow == 5: weekly[i] = 8  # Saturday peak
    else: weekly[i] = 6  # Sunday

# Strong summer peak
seasonal = 10 * np.sin(2 * np.pi * (t - 80) / 365)

# Holiday peaks
holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if (m == 12 and 20 <= d <= 31): holiday[i] = 8
    elif m == 11 and 22 <= d <= 28: holiday[i] = 5
    elif m == 3 and 10 <= d <= 20: holiday[i] = 6  # spring break
    elif m == 7 and 1 <= d <= 7: holiday[i] = 7

noise = np.random.normal(0, 2, N_POINTS)
visitors_k = base + weekly + seasonal + holiday + noise
visitors_k = np.maximum(visitors_k, 5).round(1)

df = pd.DataFrame({'date': dates, 'visitors_k': visitors_k})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,visitors_k
0,2022-01-01,19.2
1,2022-01-02,16.0
2,2022-01-03,9.6
3,2022-01-04,11.4
4,2022-01-05,7.9


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('visitors_k Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("visitors_k Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

visitors_k Statistics:
count    730.000000
mean      24.879178
std        8.595610
min        5.200000
25%       18.325000
50%       24.900000
75%       31.100000
max       51.500000
Name: visitors_k, dtype: float64

CV: 0.345
Skewness: 0.092


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:        3.9 | RMSE:        4.9 | MAPE: 18.75%
Seasonal Naive (7)             | MAE:        4.1 | RMSE:        5.3 | MAPE: 17.27%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                            Adjusted R-Squared  R-Squared       RMSE  Time Taken
Model                                                                           
KernelRidge                         345.109749 -25.469981  26.221503    0.025537
GaussianProcessRegressor             57.773206  -3.367170  10.650765    0.046388
DummyRegressor                       46.483416  -2.498724   9.533132    0.006127
QuantileRegressor                    46.431989  -2.494768   9.527741    0.062025
PassiveAggressiveRegressor           16.429869  -0.186913   5.552521    0.010045
ExtraTreeRegressor                   14.038787  -0.002984   5.104200    0.011198
OrthogonalMatchingPursuit            11.381800   0.201400   4.554549    0.009064
LassoLars                            10.856610   0.241799   4.437853    0.006952
Lasso                                10.856298   0.241823   4.437782    0.008550
MLPRegressor                         10.363780   0.279709   4.325484    0.381433


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:        2.3 | RMSE:        2.9 | MAPE: 14.24%
FLAML Test (catboost)          | MAE:        3.7 | RMSE:        4.0 | MAPE: 15.30%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        5.0 | RMSE:        5.5 | MAPE: 20.21%
SF AutoETS                     | MAE:        5.8 | RMSE:        6.4 | MAPE: 23.52%
SF SeasonalNaive               | MAE:        6.2 | RMSE:        6.7 | MAPE: 25.49%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model      MAE     RMSE      MAPE
     FLAML (catboost) 2.280968 2.863707 14.240641
FLAML Test (catboost) 3.743164 4.032310 15.304583
   Naive (Last Value) 3.935714 4.882695 18.749039
   Seasonal Naive (7) 4.121429 5.349833 17.272180
         SF AutoARIMA 4.964106 5.506970 20.209794
           SF AutoETS 5.835604 6.411559 23.516295
     SF SeasonalNaive 6.178571 6.696748 25.494863

Top 3:
                model      MAE     RMSE      MAPE
     FLAML (catboost) 2.280968 2.863707 14.240641
FLAML Test (catboost) 3.743164 4.032310 15.304583
   Naive (Last Value) 3.935714 4.882695 18.749039


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 3.44, Std: 2.10


## Interpretation and Business Insight

- Tourism demand has the strongest seasonal pattern of any service sector
- Summer accounts for the majority of annual visitors
- Weekends bring day-trippers and short-break visitors
- Holiday periods (Christmas, spring break, July 4th) create spikes
- Destination popularity growth adds a slow upward trend

## Limitations

1. Synthetic — real tourism depends on marketing, exchange rates, events
2. No source-market data (domestic vs international)
3. No weather features
4. No pricing/accommodation capacity constraints
5. Single destination — national tourism needs disaggregation

## How to Improve This Project

1. Add source-market breakdown (domestic, international by country)
2. Include exchange rate data for international tourism
3. Add weather and climate forecasts
4. Model accommodation capacity constraints
5. Include marketing spend and campaign data

## Production Considerations

- Monthly and weekly visitor forecasting for destination management
- Integration with accommodation and transport booking systems
- Environmental carrying capacity monitoring
- Marketing campaign timing optimization

## Common Mistakes

1. Ignoring source-market mix changes (different countries have different seasonality)
2. Not accounting for capacity constraints
3. Using annual totals for daily planning
4. Ignoring exchange rate effects on international tourism

## Mini Challenge / Exercises

1. Decompose into trend + seasonal + residual and analyze each
2. Model summer vs winter tourism separately
3. Add a synthetic exchange rate feature for international visitors
4. Build a monthly tourism forecast for infrastructure planning

## Final Summary / Key Takeaways

1. Tourism demand forecasting is critical for destination management
2. Seasonality is the dominant pattern (summer >> winter)
3. Weekend surges reflect day-trip and short-break demand
4. Source-market analysis is key for international destinations
5. Combining seasonal models with ML captures both structure and nuance

In [18]:
import json
metrics = {
    'project': 'Tourism Demand Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Tourism Demand Forecasting — Complete ===")

Metrics saved.

=== Tourism Demand Forecasting — Complete ===
